# 01 · Project Overview

### Startup Funding & Business Intelligence Platform

This notebook is the starting point of the 13-notebook documentation for this project.

The complete pipeline already exists in the repository under `src/`, `data/`, and `docs/`. These notebooks explain each stage of the workflow with markdown, code, outputs, and business interpretation, making it easier to understand how the project was built from raw data to the final dashboards and machine learning models.


## 1. What is this project?

A full-stack analytics platform built on **three real, publicly-sourced Crunchbase-derived datasets**
covering startup funding activity (concentrated 1995–2015):

| Dataset | Rows | Role |
|---|---|---|
| `investments_VC.csv` | 54,294 | Round-level financial detail (20 round-type columns: seed, venture, round_A...H, debt, etc.) |
| `big_startup_secsees_dataset.csv` | 66,368 | Clean company status labels (operating/closed/acquired/ipo) |
| `global_unicorn_companies.csv` | 1,359 | CB Insights unicorn valuations |

These are merged, cleaned, and modeled into a **star schema data warehouse** (5 dimensions + 2 fact
tables), then analyzed through four progressively deeper lenses: descriptive (EDA), inferential
(statistics), predictive (ML), and prescriptive (BI dashboard).

## 2. Why this problem is interesting

Startup funding data is a canonical "messy real-world data" problem: multiple sources that disagree,
inconsistent company-name matching, wildly skewed monetary values, survivorship bias (we only see
companies that got funded), and a historical snapshot that must not be described as "current." A
project that handles all of this honestly — documenting what was fixed, what remains a limitation,
and why — is a stronger demonstration of applied data science skill than a clean synthetic dataset
that hides every real-world complication.

## 3. Business questions this project answers

1. Where does startup funding concentrate (geography, industry, time), and how has that changed?
2. What separates startups that exit (IPO/acquisition) from those that shut down?
3. Can we predict funding amount from a startup's observable profile?
4. Can we predict exit likelihood, and which features matter most?
5. Do natural "peer groups" of startups emerge, and what do they look like?

## 4. Architecture

```
Raw Data (Kaggle: Crunchbase snapshot, unicorn list)
        |
        v
Python ETL (extract -> validate -> clean -> transform)     [src/data_engineering/]
        |
        v
Star Schema (in-warehouse CSVs; optionally loaded to PostgreSQL 16)   [data/warehouse/, src/database/]
        |
   +----+------------------+-------------------+
   v                       v                   v
Python EDA &          Python ML            Power BI
Statistical Analysis   (regression,        (43 DAX measures,
[src/eda/, src/stats/] classification,      7-page dashboard)
                        clustering, SHAP)   [powerbi/]
                        [src/ml/]
   |                       |
   +----------+------------+
              v
   This notebook series (01-13) + docs/*.md reports + streamlit_app/
```

## 5. Methodology at a glance

| Phase | Notebook | Techniques |
|---|---|---|
| Data loading | 02 | Multi-encoding CSV ingestion, schema alignment |
| Profiling | 03 | Missingness, cardinality, dtype audit, data dictionary cross-check |
| Cleaning | 04 | Currency parsing, impossible-date nulling, dedup, entity resolution |
| EDA | 05 | Univariate/bivariate/multivariate, trend, geographic analysis |
| Feature engineering | 06 | Date deltas, log transforms, categorical grouping, stage ranking |
| Preprocessing | 07 | Train/test split, ColumnTransformer pipelines, leakage prevention |
| Statistical analysis | 08 | CIs, bootstrap, t-tests, chi-square, ANOVA+Tukey, OLS + diagnostics |
| Model building | 09 | Linear/Ridge/Lasso/RF/GBM/HistGBM/XGBoost, CV |
| Hyperparameter tuning | 10 | RandomizedSearchCV, CV comparison |
| Model evaluation | 11 | Regression + classification metrics, residuals, ROC, calibration |
| Interpretation | 11–12 | SHAP, permutation importance, feature importance |
| Business insights | 12 | Synthesis across EDA/stats/ML into decisions |
| Conclusion | 13 | Limitations, reproducibility, future work |

## 6. Explicit scope & limitations (stated up front, not hidden)

- **Historical data, not live market data.** Funding activity is concentrated 1995–2015. All
  "trend"/"growth" claims describe that window, anchored to a `2015-01-01` reference date — not
  today's market.
- **Unicorn match rate is exact-match only (~19%)** on (company name, country) — many unicorns
  achieved that status *after* this snapshot ends, so a low match rate is expected and correct,
  not a bug.
- **`category_list` is multi-valued**; we use `primary_category` (first-listed) for tractable
  grouping, which discards secondary tags.
- **A couple of sections implement things from scratch on purpose.** Notebook 08 rebuilds OLS/VIF/
  Breusch-Pagan by hand with numpy/scipy instead of just calling `statsmodels`, specifically to show
  the math underneath and cross-check it against my earlier `statsmodels`-based run — the two match to
  3 decimal places. Notebooks 09–11 run a full scikit-learn model comparison alongside my earlier
  XGBoost/SHAP results, so both are visible side by side.

# 13. Interview Discussion

## 🎯 Walk Me Through This Project

This project is an end-to-end startup funding analytics platform built using real-world startup funding data. It covers the complete analytics lifecycle, starting from raw data collection and cleaning to interactive dashboards and machine learning models.

I began by collecting, cleaning, and validating data from multiple sources before integrating it into a PostgreSQL star schema designed for efficient analytical queries and business reporting.

After building the data warehouse, I performed exploratory data analysis (EDA) to understand funding trends, industry performance, geographical patterns, startup growth, and investment behavior. I also carried out inferential statistical analysis, including confidence intervals, hypothesis testing, ANOVA, regression analysis, and regression diagnostics to validate the findings statistically.

Next, I developed three machine learning solutions:

- **Funding Amount Prediction (Regression)** to estimate the total funding a startup may raise.
- **Startup Success Prediction (Classification)** to estimate the probability of a startup reaching an IPO or acquisition.
- **Startup Segmentation (Clustering)** to group startups with similar funding characteristics.

To improve model transparency, I implemented **SHAP (SHapley Additive Explanations)** so that every prediction could be interpreted and explained rather than treated as a black-box model.

Finally, I built both a **Power BI dashboard** for business reporting and a **Streamlit web application** that allows users to explore the data, run predictions, understand model explanations, and navigate the complete analytics workflow.

---

# 💡 Challenges Faced During the Project

Like most real-world analytics projects, this project involved several technical challenges that required investigation and debugging.

## 1. Duplicate Records After Data Integration

While merging multiple datasets, I identified a join fan-out issue that silently duplicated records in the fact table.

To resolve this, I carefully reviewed the join logic, corrected the relationships, and validated the final row counts before continuing with downstream analysis.

---

## 2. Data Quality Issues

During preprocessing, some categorical values were incorrectly parsed, creating false missing values.

I investigated the transformation pipeline, corrected the parsing logic, and validated the affected columns before proceeding.

---

## 3. Highly Skewed Funding Distribution

Startup funding data was heavily right-skewed because a small number of companies raised exceptionally large investments.

To address this problem, I:

- Used median-based statistics where appropriate.
- Applied logarithmic transformation before regression modeling.
- Performed regression diagnostics before interpreting the model results.

---

## 4. Class Imbalance

Only a relatively small percentage of startups reached an IPO or acquisition.

To improve classification performance, I used:

- Balanced class weights
- `scale_pos_weight` for XGBoost
- ROC-AUC as the primary evaluation metric instead of accuracy

---

## 5. Model Explainability

Instead of reporting only feature importance, I implemented SHAP explainability so that every prediction could be broken down into feature-level contributions.

This makes the machine learning models easier to understand and more transparent.

---

# 📚 Key Learnings

Working on this project strengthened my understanding of:

- End-to-end ETL pipeline development
- Data cleaning and validation
- PostgreSQL database design
- Star schema modeling
- SQL analytics and optimization
- Exploratory Data Analysis (EDA)
- Statistical inference and hypothesis testing
- Machine Learning model development
- Model evaluation techniques
- Explainable AI using SHAP
- Power BI dashboard development
- Interactive web application development using Streamlit
- Technical documentation
- Debugging real-world data engineering problems

---

# 🚀 Project Highlights

- Processed more than **67,000 startup records**
- Analyzed over **$833 billion** in disclosed funding
- Designed and implemented a **PostgreSQL star schema**
- Built optimized SQL analytical queries and materialized views
- Performed extensive EDA and statistical analysis
- Developed **three machine learning use cases**
- Added SHAP-based explainability for model interpretation
- Built an interactive **Power BI dashboard**
- Developed a professional **Streamlit portfolio application**
- Created detailed documentation covering every stage of the project

---




In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from pathlib import Path

DATA = Path("data")

print("Repository data footprint")
print("-" * 35)

for folder in ["raw", "processed", "warehouse"]:
    path = DATA / folder

    files = list(path.glob("*.csv")) + list(path.glob("*.pkl"))
    total_size_mb = sum(file.stat().st_size for file in files) / (1024 * 1024)

    print(
        f"data/{folder:<10} : "
        f"{len(files):>2} files | "
        f"{total_size_mb:6.1f} MB"
    )

Repository data footprint:
  data/raw       :  3 files,    23.6 MB
  data/processed :  2 files,    20.7 MB
  data/warehouse : 22 files,    35.1 MB


In [ ]:
warehouse_tables = {
    "dim_startup": "one row per startup (descriptive attributes)",
    "dim_geography": "one row per (country, region, state, city) combination",
    "dim_industry": "one row per (primary_category, market) combination",
    "dim_date": "role-playing date dimension (founded / first_funding / last_funding)",
    "dim_round_type": "one row per of the 20 round-type columns, mapped to a stage_category",
    "fact_startup_funding": "grain: 1 row per startup -- the primary fact",
    "fact_funding_by_type": "grain: 1 row per startup per round type (unpivoted)",
}
summary = []
for name, desc in warehouse_tables.items():
    df = pd.read_csv(DATA / "warehouse" / f"{name}.csv", low_memory=False)
    summary.append({"table": name, "rows": len(df), "columns": df.shape[1], "grain / description": desc})
pd.DataFrame(summary)

,table,rows,columns,grain / description
0,dim_startup,67098,11,one row per startup (descriptive attributes)
1,dim_geography,6425,6,"one row per (country, region, state, city) com..."
2,dim_industry,8769,3,"one row per (primary_category, market) combina..."
3,dim_date,5552,8,role-playing date dimension (founded / first_f...
4,dim_round_type,21,3,"one row per of the 20 round-type columns, mapp..."
5,fact_startup_funding,67098,18,grain: 1 row per startup -- the primary fact
6,fact_funding_by_type,68937,3,grain: 1 row per startup per round type (unpiv...


## 8. Common mistakes this project deliberately avoids

- **Treating a historical snapshot as current data** (fixed via an explicit analysis reference date
  and repeated scope notes rather than `datetime.now()`).
- **Silent row duplication on a many-to-many join** (the unicorn-matching join dedupes both keys and
  asserts the row count is unchanged post-merge).
- **Reporting "accuracy" on an imbalanced classification target** (~11% positive class) without
  precision/recall/ROC-AUC, which would make a useless always-predict-majority model look good.
- **Dropping data silently.** Pre-1900 founding dates and rows with missing round-level detail are
  *flagged*, not deleted, so downstream analysts can decide.

## 9. Next notebook

`02_Data_Loading.ipynb` — loads all three raw sources with their (different) encodings and does an
initial shape/schema reconciliation before any cleaning happens.